In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime
from typing import Any
from dataclasses import dataclass, field

import anndata as ad

LOG_KEY = "_omicslog"


def _timestamp() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def _format_log_message(operation: str, message: str, ts: str | None = None) -> str:
    stamp = ts or _timestamp()
    return f"[{stamp}] {operation}: {message}"


def _ensure_log_container(adata: ad.AnnData) -> list[str]:
    current = adata.uns.get(LOG_KEY)
    if not isinstance(current, list):
        adata.uns[LOG_KEY] = []
    return adata.uns[LOG_KEY]


def _append_log_messages(adata: ad.AnnData, messages: list[str] | tuple[str, ...]) -> None:
    if not messages:
        return
    _ensure_log_container(adata).extend(messages)


def _inherit_and_append(
    parent: ad.AnnData,
    child: ad.AnnData,
    messages: list[str] | tuple[str, ...],
) -> None:
    child.uns[LOG_KEY] = list(_ensure_log_container(parent))
    _append_log_messages(child, messages)


def _subset_messages(
    pre: AnnDataSnapshot,
    post: AnnDataSnapshot,
    operation: str = "subset",
) -> list[str]:
    msgs: list[str] = []
    ts = _timestamp()

    if pre.n_vars != post.n_vars:
        removed_genes = pre.n_vars - post.n_vars
        percent_removed = round((removed_genes / pre.n_vars) * 100) if pre.n_vars else 0
        msgs.append(
            _format_log_message(
                operation,
                f"removed {removed_genes} genes ({percent_removed}%), {post.n_vars} genes remaining",
                ts,
            )
        )

    if pre.n_obs != post.n_obs:
        removed_samples = pre.n_obs - post.n_obs
        percent_removed = round((removed_samples / pre.n_obs) * 100) if pre.n_obs else 0
        msgs.append(
            _format_log_message(
                operation,
                f"removed {removed_samples} samples ({percent_removed}%), {post.n_obs} samples remaining",
                ts,
            )
        )
    
    if pre.layers != post.layers:
        new_layer = list(post.layers)[-1] if post.layers else None
        msgs.append(
            _format_log_message(
                operation,
                f"added layers: {', '.join([new_layer])} layer added",
                ts,
            )
        )
    
    return msgs


@dataclass
class AnnDataSnapshot:
    """Captures the full state of an AnnData object for diffing."""
    n_obs: int
    n_vars: int
    obs_cols: list[str] = field(default_factory=list)
    var_cols: list[str] = field(default_factory=list)
    layers: list[str] = field(default_factory=list)
    obsm: list[str] = field(default_factory=list)
    varm: list[str] = field(default_factory=list)
    obsp: list[str] = field(default_factory=list)
    varp: list[str] = field(default_factory=list)

    @classmethod
    def from_anndata(cls, adata: ad.AnnData) -> "AnnDataSnapshot":
        return cls(
            n_obs=adata.n_obs,
            n_vars=adata.n_vars,
            obs_cols=list(adata.obs.columns),
            var_cols=list(adata.var.columns),
            layers=list(adata.layers.keys()),
            obsm=list(adata.obsm.keys()),
            varm=list(adata.varm.keys()),
            obsp=list(adata.obsp.keys()),
            varp=list(adata.varp.keys()),
        )

class LoggedAnnDataStandalone(ad.AnnData):
    """Standalone subclass strategy with local logging helpers and message style."""

    def __init__(self, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        _ensure_log_container(self)

    @classmethod
    def _safe_component_copy(cls, value):
        return value.copy() if hasattr(value, "copy") else deepcopy(value)

    @classmethod
    def from_anndata(cls, adata: ad.AnnData) -> "LoggedAnnDataStandalone":
        if isinstance(adata, cls):
            _ensure_log_container(adata)
            return adata

        kwargs: dict[str, Any] = {
            "X": cls._safe_component_copy(adata.X) if adata.X is not None else None,
            "obs": adata.obs.copy(),
            "var": adata.var.copy(),
            "uns": deepcopy(dict(adata.uns)),
            "obsm": {k: cls._safe_component_copy(v) for k, v in adata.obsm.items()},
            "varm": {k: cls._safe_component_copy(v) for k, v in adata.varm.items()},
            "layers": {k: cls._safe_component_copy(v) for k, v in adata.layers.items()},
            "obsp": {k: cls._safe_component_copy(v) for k, v in adata.obsp.items()},
            "varp": {k: cls._safe_component_copy(v) for k, v in adata.varp.items()},
        }
        if adata.raw is not None:
            kwargs["raw"] = {
                "X": cls._safe_component_copy(adata.raw.X),
                "var": adata.raw.var.copy(),
                "varm": {k: cls._safe_component_copy(v) for k, v in adata.raw.varm.items()},
            }

        logged = cls(**kwargs)
        _ensure_log_container(logged)
        return logged

    def _snapshot(self) -> AnnDataSnapshot:
        return AnnDataSnapshot.from_anndata(self)
    
    def __getitem__(self, index):
        pre_shape = self._snapshot()
        result = super().__getitem__(index)
        logged_result = self.from_anndata(result)
        msgs = _subset_messages(pre_shape, logged_result._snapshot(), operation="subset")
        _inherit_and_append(self, logged_result, msgs)
        return logged_result

    def _inplace_subset(self, index):
        pre_shape = self._snapshot()
        super()._inplace_subset(index)
        _append_log_messages(self, _subset_messages(pre_shape, self._snapshot(), operation="subset"))

    def _operation_log_block(self) -> str:
        logs = self.uns.get(LOG_KEY, [])
        if not logs:
            return ""
        return "\n\nOperation log:\n" + "\n".join(str(x) for x in logs)

    def __repr__(self) -> str:
        return super().__repr__() + self._operation_log_block()

    def __str__(self) -> str:
        return self.__repr__()

    def operation_log(self) -> list[str]:
        return list(self.uns.get(LOG_KEY, []))


def log_start(adata: ad.AnnData) -> LoggedAnnDataStandalone:
    return LoggedAnnDataStandalone.from_anndata(adata)


In [2]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
print(ad.__version__)

counts = csr_matrix(np.random.poisson(1, size=(100, 2000)), dtype=np.float32)
adata = ad.AnnData(counts)

adata.obs_names = [f"Cell_{i:d}" for i in range(adata.n_obs)]
adata.var_names = [f"Gene_{i:d}" for i in range(adata.n_vars)]

logdata = log_start(adata)
print(logdata)

ct = np.random.choice(["B", "T", "Monocyte"], size=(logdata.n_obs,))
logdata.obs["cell_type"] = pd.Categorical(ct)  # Categoricals are preferred for efficiency
print(logdata.obs)

0.12.10
AnnData object with n_obs × n_vars = 100 × 2000
    uns: '_omicslog'
        cell_type
Cell_0          B
Cell_1          B
Cell_2   Monocyte
Cell_3   Monocyte
Cell_4   Monocyte
...           ...
Cell_95         B
Cell_96         T
Cell_97         B
Cell_98         T
Cell_99  Monocyte

[100 rows x 1 columns]


/tmp/ipykernel_45318/1596706344.py:5: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(ad.__version__)


'log_transformed'

# TODO (24/03/17)

1. Fix the example to improve reproducibility
2. add and .vars based filtration
3. Fix the problem regarding Layers comparisons


# Filtrating by cells

In [3]:
log1 = logdata[["Cell_1", "Cell_10"], ]
log2 = logdata[["Cell_1", "Cell_10"], ].copy()

INSIDE THE FUNCTION
INSIDE THE FUNCTION


In [4]:
print("ORIGINAL VERSION")
print(logdata)

print("PASSED VERSIONS")
print(log1)

print("COPIED VERSION")
print(log2)

ORIGINAL VERSION
AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: '_omicslog'
PASSED VERSIONS
AnnData object with n_obs × n_vars = 2 × 2000
    obs: 'cell_type'
    uns: '_omicslog'

Operation log:
[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining
COPIED VERSION
AnnData object with n_obs × n_vars = 2 × 2000
    obs: 'cell_type'
    uns: '_omicslog'


In [5]:
print("ORIGINAL VERSION")
print(logdata.uns)

print("PASSED VERSIONS")
print(log1.uns)

print("COPIED VERSION")
print(log2.uns)

ORIGINAL VERSION
OrderedDict([('_omicslog', [])])
PASSED VERSIONS
{'_omicslog': ['[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining']}
COPIED VERSION
{'_omicslog': ['[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining']}


In [6]:
print(log1.is_view)
print(log2.is_view)

False
False


# Fltrating by Cells (.obs)

In [7]:
log1 = log1[log1.obs.cell_type == "B"]
log1.uns

INSIDE THE FUNCTION


{'_omicslog': ['[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining']}

In [8]:
log2 = log2[log2.obs.cell_type == "B"]
log2.uns

{'_omicslog': ['[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining']}

In [9]:
print(log1.shape)
print(log2.shape)

(2, 2000)
(2, 2000)


# Adding observatons and variables

In [23]:
log1.obsm["X_umap"] = np.random.normal(0, 1, size=(log1.n_obs, 2))
log1.varm["gene_stuff"] = np.random.normal(0, 1, size=(log1.n_vars, 5))
log1.uns

{'_omicslog': ['[2026-03-17 09:40:58] subset: removed 98 samples (98%), 2 samples remaining',
  '[2026-03-17 09:41:09] subset: removed 1 samples (50%), 1 samples remaining']}

In [24]:
log2.obsm["X_umap"] = np.random.normal(0, 1, size=(log2.n_obs, 2))
log2.varm["gene_stuff"] = np.random.normal(0, 1, size=(log2.n_vars, 5))
log2.uns

/tmp/ipykernel_39509/2545183977.py:1: ImplicitModificationWarning: Setting element `.obsm['X_umap']` of view, initializing view as actual.
  log2.obsm["X_umap"] = np.random.normal(0, 1, size=(log2.n_obs, 2))


{'_omicslog': ['[2026-03-17 09:40:58] subset: removed 98 samples (98%), 2 samples remaining']}

# Adding layers

In [10]:
log1.layers["log_transformed"] = np.log1p(log1.X)
log2.layers["log_transformed"] = np.log1p(log2.X)

print(log1)
print(log2)

AnnData object with n_obs × n_vars = 2 × 2000
    obs: 'cell_type'
    uns: '_omicslog'
    layers: 'log_transformed'

Operation log:
[2026-03-17 09:56:41] subset: removed 98 samples (98%), 2 samples remaining
AnnData object with n_obs × n_vars = 2 × 2000
    obs: 'cell_type'
    uns: '_omicslog'
    layers: 'log_transformed'


/tmp/ipykernel_45318/3334769030.py:2: ImplicitModificationWarning: Setting element `.layers['log_transformed']` of view, initializing view as actual.
  log2.layers["log_transformed"] = np.log1p(log2.X)


In [11]:
print(log1.uns)
print(log2.uns)

{'_omicslog': ['[2026-03-17 09:43:52] subset: removed 98 samples (98%), 2 samples remaining']}
{'_omicslog': ['[2026-03-17 09:43:52] subset: removed 98 samples (98%), 2 samples remaining']}


# Export/Import

In [15]:
log1.write('log1.h5ad', compression="gzip")

In [ ]:
log1 = ad.read_h5ad('log1.h5ad')
print(log1)

AnnData object with n_obs × n_vars = 0 × 2000
    obs: 'cell_type'
    uns: '_omicslog'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'


In [17]:
print(log1.uns)

{'_omicslog': array(['[2026-03-17 07:50:44] subset: removed 98 samples (98%), 2 samples remaining',
       '[2026-03-17 07:51:00] subset: removed 2 samples (100%), 0 samples remaining'],
      dtype=object)}
